# Heron 101 Fine-Tuning on Construction Documents

This notebook runs RT-DETR fine-tuning for Heron 101 on construction document pages with a focus on improving bounding box placement.

Current training setup:
- 4x `A100-80GB`
- train batch size: `16`
- val batch size: `16`
- main learning rate: `0.00015`
- backbone learning rate: `0.000015`
- epochs: `200`
- warmup: `100`
- seed: `3407`
- `aux_loss=True`
- `num_denoising=100`
- class-agnostic bbox evaluation on a 17-class construction document dataset

Run cells with `Shift+Enter`. You can update the repo branch, output run name, or config path below as needed.

In [ ]:
%uv pip install transformers Pillow torch requests
%uv pip install modal
%uv pip install pycocotools
%uv pip install faster-coco-eval pycocotools scipy pyyaml opencv-python-headless matplotlib
import modal
import os
import shutil
from pathlib import Path
os.environ["TORCH_HOME"] = "/outputs/.torch"
os.environ["HF_HOME"] = "/outputs/.hf"
os.environ["PYTHONPYCACHEPREFIX"] = "/outputs/.pycache"

# (Optional) sanity check
print("modal_version", getattr(modal, "__version__", "unknown"))

In [ ]:
# Fill this in after pushing your repo.
# Examples:
#   REPO_URL = "https://github.com/<you>/RT-DETR.git"
#   REPO_URL = "git@github.com:<you>/RT-DETR.git"  (if you set up SSH keys in Modal)
REPO_URL = "https://github.com/neelimagprasad/RT-DETR.git"
REPO_BRANCH = "neelima/bbox-data-training"
DATA_VOL_NAME = "rtdetr-bbox-data"
OUT_VOL_NAME = "rtdetr-outputs"

# GPU options in Modal:
# - modal.gpu.A10G()
# - modal.gpu.A100()
# - modal.gpu.A100(count=4)  # multi-GPU in one container
GPU = "A100-80GB:4"

# Long-running training: increase as needed
TIMEOUT_SECS = 60 * 60 * 12  # 12 hours
!git clone --depth 1 --branch neelima/bbox-data-training --single-branch https://github.com/neelimagprasad/RT-DETR.git /root/RT-DETR
repo_dir = Path("/root/RT-DETR")


In [5]:
#Command to run training
app = modal.App("rtdetrv2-heron101-train_aux")

data_vol = modal.Volume.from_name(DATA_VOL_NAME, create_if_missing=True)
out_vol = modal.Volume.from_name(OUT_VOL_NAME, create_if_missing=True)

# GPU image with CUDA torch + training deps.
# Note: you can move more installs into this image over time to speed up iteration.
image = (
    modal.Image.from_registry(
        "nvidia/cuda:12.1.1-cudnn8-runtime-ubuntu22.04",
        add_python="3.12",
    )
    .apt_install("git", "libglib2.0-0", "libgl1")
    .run_commands(
        "python -m pip install -U pip",
        "python -m pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121",
        "python -m pip install opencv-python-headless scipy pycocotools pyyaml tqdm huggingface_hub safetensors matplotlib",
    )
)


def _run(cmd, *, cwd=None, env=None, stream: bool = True):
    """Run a command; stream output live by default."""
    import subprocess

    print("$", " ".join(cmd), flush=True)

    if not stream:
        p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout, flush=True)
        if p.stderr:
            print(p.stderr, flush=True)
        if p.returncode != 0:
            raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout, stderr=p.stderr)
        return p

    # Stream stdout+stderr in real time, but still keep a tail for error reporting.
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert proc.stdout is not None
    tail = []
    tail_keep = 400  # keep last N lines
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > tail_keep:
            tail = tail[-tail_keep:]

    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd, output="".join(tail), stderr="".join(tail))
    return rc


@app.function(
    gpu=GPU,
    timeout=TIMEOUT_SECS,
    image=image,
    volumes={
        "/data": data_vol,
        "/outputs": out_vol,
    },
)
def train_from_heron101(
    *,
    run_name: str = "run1",
    cfg_path: str = "rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r101vd_bbox_data_heron101.yml",

):
    

    # Ensure we are on the branch that contains your converter + configs.
    if not repo_dir.exists():
        _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(repo_dir)])
    else:
        _run(["git", "fetch", "origin", REPO_BRANCH], cwd=str(repo_dir))
        _run(["git", "checkout", REPO_BRANCH], cwd=str(repo_dir))
        _run(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"] , cwd=str(repo_dir))

    # Make sure your dataset is present (uploaded to the Volume)
    src_data = Path("/data/bbox_data")
    if not src_data.exists():
        raise RuntimeError("/data/bbox_data not found. Did you upload it to the Modal volume?")

    dst_data = repo_dir / "rtdetrv2_pytorch" / "dataset" / "bbox_data"
    if dst_data.exists():
        shutil.rmtree(dst_data)
    shutil.copytree(src_data, dst_data)

    # Install repo requirements if present
    req = repo_dir / "rtdetrv2_pytorch" / "requirements.txt"
    if req.exists():
        _run(["python", "-m", "pip", "install", "-r", str(req)], cwd=str(repo_dir))

    # Convert Heron-101 HF weights -> repo-compatible .pth (cached in /outputs)
    converted = Path("/outputs/heron101_converted_with_backbone.pth")
    if not converted.exists():
        _run(
            [
                "python",
                "rtdetrv2_pytorch/tools/convert_hf_heron101_to_rtdetrv2.py",
                "--config",
                cfg_path,
                "--output",
                str(converted),
            ],
            cwd=str(repo_dir),
        )

    out_dir = Path("/outputs") / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Preflight: confirm GPU + paths before training
    _run(["python", "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('cuda_count', torch.cuda.device_count()); print('cuda_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"])
    _run(["bash", "-lc", "nvidia-smi || true"])
    _run(["bash", "-lc", "ls -la /data && ls -la /data/bbox_data && ls -la /outputs | head"])

    # If you requested multiple GPUs (e.g. "A100-80GB:2"), launch DDP with torchrun.
    # Rely on the requested GPU string rather than CUDA_VISIBLE_DEVICES.
    if isinstance(GPU, str) and ":" in GPU:
        try:
            nproc = int(GPU.split(":", 1)[1])
        except Exception:
            nproc = 1
    else:
        nproc = 1

    # Build the training command.
    # Note: this repo always writes `last.pth` every epoch and updates `best.pth` when improved.
    # `checkpoint_freq=10` only controls the extra numbered snapshots like checkpoint0009.pth.
    if nproc > 1:
        train_cmd = [
            "torchrun",
            "--standalone",
            f"--nproc_per_node={nproc}",
            "rtdetrv2_pytorch/tools/train.py",
            "-c",
            cfg_path,
            "-t",
            str(converted),
            #"-r",
            #"/outputs/heron101_bbox_data_run11/last.pth",
            "--output-dir",
            str(out_dir),
            "--seed",
            "3407",
            "-u",
            # 4-GPU batch sizing (or generally: world-size batch sizing)
            f"train_dataloader.total_batch_size=16",
            f"val_dataloader.total_batch_size=16",
            # more epochs + much shorter warmup so multi-GPU doesn't spend the whole run in warmup
            "epoches=200",
            "checkpoint_freq=10",
            "lr_warmup_scheduler.warmup_duration=100",
            # DDP needs this because stability mode intentionally leaves some aux params unused
            "find_unused_parameters=True",
            # stability mode for very large document pages
            "use_amp=False",
            "RTDETRTransformerv2.aux_loss=True",
            "RTDETRTransformerv2.num_denoising=100",
            "train_dataloader.dataset.transforms.policy.epoch=0",
            "train_dataloader.collate_fn.scales=~",
            "train_dataloader.collate_fn.stop_epoch=0",
            # logging
            "print_freq=20",
        ]
    else:
        train_cmd = [
            "python",
            "rtdetrv2_pytorch/tools/train.py",
            "-c",
            cfg_path,
            "-t",
            str(converted),
            "--device",
            "cuda",
            "--output-dir",
            str(out_dir),
            "-u",
            "epoches=60",
            "checkpoint_freq=10",
            # stability mode for very large document pages
            "use_amp=False",
            "RTDETRTransformerv2.aux_loss=False",
            "RTDETRTransformerv2.num_denoising=0",
            "train_dataloader.dataset.transforms.policy.epoch=0",
            "train_dataloader.collate_fn.scales=~",
            "train_dataloader.collate_fn.stop_epoch=0",
            "print_freq=20",
        ]

    # Stream logs + write them to the outputs volume.
    log_file = out_dir / "train.log"
    env_prefix = "PYTHONUNBUFFERED=1 TORCH_DISTRIBUTED_DEBUG=DETAIL NCCL_DEBUG=INFO"
    shell_cmd = env_prefix + " " + " ".join(train_cmd) + f" 2>&1 | tee -a {log_file}"
    _run(["bash", "-lc", shell_cmd], cwd=str(repo_dir))

    # Persist volume writes
    out_vol.commit()
    data_vol.commit()

    return {
        "output_dir": str(out_dir),
        "converted_checkpoint": str(converted),
        "nproc": nproc,
    }

In [6]:
with app.run():
    result = train_from_heron101.remote(run_name="heron101_bbox_data_run18")
    print(result)

In [ ]:
# Command to run eval on your checkpoint. Replace (`run_name` and checkpoint path as needed.)
!cd /root/RT-DETR && python rtdetrv2_pytorch/tools/eval_and_visualize.py \
  -c rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r101vd_bbox_data_heron101.yml \
  -r /mnt/rtdetr-outputs/heron101_bbox_data_run8/best.pth \
  --images-dir /mnt/rtdetr-bbox-data/bbox_data/images_test \
  --coco-json /mnt/rtdetr-bbox-data/bbox_data/annotations/instances_test.json \
  --device cuda \
  --save-vis-dir /mnt/rtdetr-outputs/heron101_bbox_data_run5/eval_vis_test_ca_nms \
  --results-json /mnt/rtdetr-outputs/heron101_bbox_data_run5/eval_results_test_ca_nms.json \
  --metrics-json /mnt/rtdetr-outputs/heron101_bbox_data_run5/eval_metrics_test_ca_nms.json \
  --class-agnostic-eval \
  --nms-iou-threshold 0.5 \
  --eval-score-threshold 0.05 \
  --vis-score-threshold 0.5

In [ ]:
# Command to Generate loss curves from the log.txt file in your training folder
!cd /root/RT-DETR && python3 rtdetrv2_pytorch/tools/plot_training_log.py /mnt/rtdetr-outputs/heron101_bbox_data_run8/log.txt

In [ ]:
# Command to convert your .pth checkpoint to .onnx
!python3 /root/RT-DETR/rtdetrv2_pytorch/tools/export_onnx.py \
  -c /root/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r101vd_bbox_data_heron101.yml \
  -r /mnt/rtdetr-outputs/heron101_bbox_data_run8/best.pth \
  -o /mnt/rtdetr-outputs/heron101_bbox_data_run8/best.onnx \
  -s 640 \
  --check \
  --simplify

In [ ]:
!python3 -m pip install -r /root/RT-DETR/rtdetrv2_pytorch/requirements.txt